In [1]:
#!/usr/bin/env python3
"""
Agentic checkpoint evaluation pipeline for near-tied MLLM checkpoints.

Designed to match the paper/rebuttal workflow:
1) Optional construction of evaluation JSON from a PDF with one sample/page.
2) Pointwise scoring for all candidate checkpoints.
3) Adaptive filtering of clearly inferior candidates.
4) Listwise ranking of remaining candidates with randomized/anonymized ordering.
5) Pairwise comparison of unresolved finalists.
6) Bootstrap / repeated-subsampling stability analysis.
7) Percentile-based stability score with beta/gamma ablation.
8) Optional validation-loss baseline.
9) Optional repeated-generation stochasticity analysis if replicate generations exist.

IMPORTANT:
- Keep all exact protocol constants configurable until recovered from the original code/logs.
- Do not claim statistics computed on the 100-sample public subset were computed on the full ~2,400 examples.
- This script does NOT hard-code proprietary API credentials or current vendor model IDs.
"""

from __future__ import annotations

import argparse
import base64
import hashlib
import json
import math
import os
import random
import re
import statistics
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------

@dataclass
class EvalConfig:
    seed: int = 42

    # Public reproducibility subset.
    public_subset_size: int = 100

    # Original paper mentions repeated 800-example subsets from ~2,400.
    # Recover the exact original R / replacement rule before claiming them.
    original_subsample_size: int = 800
    original_resampling_rounds: Optional[int] = None  # XX
    original_with_replacement: Optional[bool] = None  # XX

    # New rebuttal/public-100 bootstrap analysis.
    bootstrap_rounds: int = 1000
    bootstrap_with_replacement: bool = True

    # Stage I -> II and Stage II -> III thresholds.
    # XX until recovered; defaults below are experimentation values only.
    pointwise_keep_margin: float = 0.10
    pointwise_min_candidates: int = 4
    pointwise_max_candidates: int = 6

    listwise_finalists: int = 2
    pairwise_confidence_threshold: float = 0.70

    # Percentile score:
    # S = P50 - beta*(P50-P20) + gamma*(P80-P50)
    # Initial ablation defaults, NOT claimed paper values.
    beta: float = 0.50
    gamma: float = 0.25

    beta_grid: Tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0)
    gamma_grid: Tuple[float, ...] = (0.0, 0.1, 0.25, 0.5)

    # Randomize candidate display order to mitigate position bias.
    randomize_candidate_order: bool = True

    # Judge names are labels only. Replace with exact API model IDs.
    pointwise_judge_label: str = "Gemini 3 Flash"
    listwise_judge_label: str = "Claude Sonnet 4.1"
    pairwise_judge_label: str = "GPT-4o reasoning"

    # Keep actual API temperature values configurable.
    pointwise_judge_temperature: Optional[float] = None  # XX
    listwise_judge_temperature: Optional[float] = None   # XX
    pairwise_judge_temperature: Optional[float] = None   # XX

In [2]:
# ---------------------------------------------------------------------
# DATA FORMAT
# ---------------------------------------------------------------------

"""
Expected metadata JSONL item:

{
  "sample_id": "sample_0001",
  "page_number": 1,
  "question": "look at and read the first and second emails.",
  "image_path": "/content/drive/MyDrive/.../sample_0001.jpg",
  "responses": {
      "dpo_enhin_I_2000": "...",
      "dpo_enhin_I_4000": "...",
      ...
  },

  # Optional repeated generations for stochasticity analysis:
  "response_replicates": {
      "dpo_enhin_I_2000": ["gen1", "gen2", "gen3"],
      ...
  },

  # Optional validation/training-loss baseline:
  "training_loss": {
      "dpo_enhin_I_2000": 1.23,
      ...
  }
}

A separate mapping JSON may contain:
{
  "dpo_enhin_I_2000": {"iteration": 2000, "anonymous_id": "ckpt_A"},
  ...
}
"""


# ---------------------------------------------------------------------
# BASIC IO
# ---------------------------------------------------------------------

def read_jsonl(path: str | Path) -> List[Dict[str, Any]]:
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def write_jsonl(rows: Iterable[Dict[str, Any]], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_json(obj: Any, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]


# ---------------------------------------------------------------------
# OPTIONAL PDF -> PAGE TEXT / PAGE IMAGE CONSTRUCTION
# ---------------------------------------------------------------------

def render_pdf_pages(pdf_path: str | Path, out_dir: str | Path, dpi: int = 160) -> List[Path]:
    """
    Render one PNG per PDF page using PyMuPDF.
    Useful when each PDF page corresponds to exactly one evaluation sample.
    """
    if fitz is None:
        raise ImportError("Install PyMuPDF: pip install pymupdf")
    pdf_path = Path(pdf_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(pdf_path)
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    paths = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(matrix=mat, alpha=False)
        out = out_dir / f"page_{i+1:04d}.png"
        pix.save(out)
        paths.append(out)
    return paths


def extract_pdf_page_text(pdf_path: str | Path) -> List[str]:
    """
    Uses embedded PDF text only; no OCR.
    PPT->PDF exports normally preserve text and should work well.
    """
    if fitz is None:
        raise ImportError("Install PyMuPDF: pip install pymupdf")
    doc = fitz.open(pdf_path)
    return [page.get_text("text") for page in doc]


CKPT_HEADER_RE = re.compile(
    r"(?P<ckpt>(?:dpo|sft|ckpt)[\w\-\.]*?(?:_I_)?\d+)\s*:\s*",
    flags=re.IGNORECASE
)


def parse_page_text_heuristic(text: str) -> Dict[str, Any]:
    """
    Heuristic parser for slides/pages shaped like the provided example.

    It tries to extract:
    - question after 'Question:'
    - blocks headed by checkpoint IDs ending in ':'

    Review output manually before publication.
    """
    clean = re.sub(r"\r", "\n", text)
    clean = re.sub(r"\n{3,}", "\n\n", clean).strip()

    q_match = re.search(r"Question\s*:\s*(.+?)(?:\n|$)", clean, flags=re.I)
    question = q_match.group(1).strip() if q_match else ""

    matches = list(CKPT_HEADER_RE.finditer(clean))
    responses: Dict[str, str] = {}

    for i, m in enumerate(matches):
        ckpt = m.group("ckpt").strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(clean)
        response = clean[start:end].strip()
        responses[ckpt] = response

    return {
        "question": question,
        "responses": responses,
        "raw_page_text": clean,
    }


def build_json_from_pdf_and_image_folder(
    pdf_path: str | Path,
    image_folder: str | Path,
    output_jsonl: str | Path,
    image_glob: str = "*",
) -> List[Dict[str, Any]]:
    """
    Assumption:
      PDF page 1 <-> first sorted image
      PDF page 2 <-> second sorted image
      ...
    Use only if this alignment is actually true.

    The image folder can be a mounted Google Drive directory in Colab.
    """
    texts = extract_pdf_page_text(pdf_path)
    images = sorted([p for p in Path(image_folder).glob(image_glob) if p.is_file()])

    if len(images) != len(texts):
        raise ValueError(
            f"Page/image mismatch: {len(texts)} PDF pages vs {len(images)} images. "
            "Do not continue until alignment is verified."
        )

    rows = []
    for idx, (text, image_path) in enumerate(zip(texts, images), start=1):
        parsed = parse_page_text_heuristic(text)
        rows.append({
            "sample_id": f"sample_{idx:04d}",
            "page_number": idx,
            "question": parsed["question"],
            "image_path": str(image_path),
            "responses": parsed["responses"],
            "raw_page_text": parsed["raw_page_text"],
        })

    write_jsonl(rows, output_jsonl)
    return rows



In [ ]:
# ---------------------------------------------------------------------
# MODEL MAPPING / ANONYMIZATION
# ---------------------------------------------------------------------

def checkpoint_iteration(name: str) -> Optional[int]:
    nums = re.findall(r"(\d+)", name)
    return int(nums[-1]) if nums else None


def build_checkpoint_mapping(
    samples: Sequence[Dict[str, Any]],
    out_path: Optional[str | Path] = None,
    seed: int = 42,
) -> Dict[str, Dict[str, Any]]:
    all_ckpts = sorted({k for s in samples for k in s.get("responses", {}).keys()})
    rng = random.Random(seed)
    anon_names = [f"ckpt_{chr(ord('A') + i)}" for i in range(len(all_ckpts))]
    rng.shuffle(anon_names)

    mapping = {}
    for ckpt, anon in zip(all_ckpts, anon_names):
        mapping[ckpt] = {
            "anonymous_id": anon,
            "iteration": checkpoint_iteration(ckpt),
        }

    if out_path:
        save_json(mapping, out_path)
    return mapping


def shuffled_candidate_order(
    checkpoint_names: Sequence[str],
    sample_id: str,
    stage: str,
    seed: int,
) -> List[str]:
    # Deterministic but independently shuffled per sample/stage.
    h = int(hashlib.sha256(f"{seed}|{sample_id}|{stage}".encode()).hexdigest()[:8], 16)
    rng = random.Random(h)
    arr = list(checkpoint_names)
    rng.shuffle(arr)
    return arr


In [ ]:
# ---------------------------------------------------------------------
# RUBRICS / PROMPT BUILDERS
# ---------------------------------------------------------------------

POINTWISE_RUBRIC = """
Evaluate the candidate response to the image/question.

Dimensions:
1. Factual correctness and support from visible evidence.
2. Relevance to the user's question.
3. Usefulness/completeness without unnecessary content.
4. Visual/OCR grounding.
5. Hallucination: penalize unsupported claims.

Groundedness takes precedence over stylistic fluency.

Return strict JSON:
{
  "score": <float from 0 to 1>,
  "factuality": <0 to 1>,
  "relevance": <0 to 1>,
  "usefulness": <0 to 1>,
  "grounding": <0 to 1>,
  "hallucination_penalty": <0 to 1>,
  "rationale": "<brief>"
}
""".strip()


LISTWISE_RUBRIC = """
Rank all anonymous candidate responses to the SAME image/question.

Primary criteria:
1. factual/visual correctness,
2. relevance,
3. usefulness,
4. absence of hallucination,
5. OCR/visual grounding.

Do not infer checkpoint quality from candidate labels or ordering.

Return strict JSON:
{
  "ranking": ["C3", "C1", "C2", ...],
  "rationale": {"C3": "...", "C1": "...", ...}
}
""".strip()


PAIRWISE_RUBRIC = """
Choose which anonymous response better answers the question from the image.

Prioritize:
- factual/visual correctness,
- relevance,
- usefulness,
- OCR/visual grounding,
- absence of hallucination.

Return strict JSON:
{
  "winner": "A" or "B" or "TIE",
  "confidence": <0 to 1>,
  "rationale": "<brief>"
}
""".strip()

In [ ]:
# ---------------------------------------------------------------------
# API ABSTRACTION
# ---------------------------------------------------------------------

class JudgeClient:
    """
    Plug in your actual Gemini / Anthropic / OpenAI API calls here.

    The pipeline intentionally depends only on this JSON interface so the
    statistical analysis is vendor-agnostic and reproducible.
    """

    def pointwise(
        self,
        image_path: str,
        question: str,
        response: str,
        model_label: str,
        temperature: Optional[float],
    ) -> Dict[str, Any]:
        raise NotImplementedError

    def listwise(
        self,
        image_path: str,
        question: str,
        candidates: Dict[str, str],
        model_label: str,
        temperature: Optional[float],
    ) -> Dict[str, Any]:
        raise NotImplementedError

    def pairwise(
        self,
        image_path: str,
        question: str,
        response_a: str,
        response_b: str,
        model_label: str,
        temperature: Optional[float],
    ) -> Dict[str, Any]:
        raise NotImplementedError

In [ ]:
class MockJudgeClient(JudgeClient):
    """
    Dry-run client.
    Produces deterministic pseudo-results so you can validate the pipeline
    before spending API calls.
    """

    @staticmethod
    def _u(text: str) -> float:
        h = int(hashlib.sha256(text.encode()).hexdigest()[:8], 16)
        return (h % 10000) / 10000.0

    def pointwise(self, image_path, question, response, model_label, temperature):
        u = self._u(question + "|" + response)
        score = 0.3 + 0.65 * u
        return {
            "score": round(score, 4),
            "factuality": round(score, 4),
            "relevance": round(min(1, score + 0.03), 4),
            "usefulness": round(score, 4),
            "grounding": round(max(0, score - 0.02), 4),
            "hallucination_penalty": round(1 - score, 4),
            "rationale": "MOCK",
        }

    def listwise(self, image_path, question, candidates, model_label, temperature):
        scores = {
            cid: self._u(question + "|" + txt + "|listwise")
            for cid, txt in candidates.items()
        }
        ranking = sorted(scores, key=scores.get, reverse=True)
        return {
            "ranking": ranking,
            "rationale": {k: "MOCK" for k in ranking},
        }

    def pairwise(self, image_path, question, response_a, response_b, model_label, temperature):
        a = self._u(question + "|" + response_a + "|pairwise")
        b = self._u(question + "|" + response_b + "|pairwise")
        if abs(a - b) < 0.02:
            winner = "TIE"
        else:
            winner = "A" if a > b else "B"
        return {
            "winner": winner,
            "confidence": round(abs(a - b), 4),
            "rationale": "MOCK",
        }


In [ ]:
# ---------------------------------------------------------------------
# STAGE I: POINTWISE
# ---------------------------------------------------------------------

def run_pointwise(
    samples: Sequence[Dict[str, Any]],
    judge: JudgeClient,
    cfg: EvalConfig,
    cache_path: Optional[str | Path] = None,
) -> pd.DataFrame:
    rows = []

    for sample in samples:
        sid = sample["sample_id"]
        ckpts = list(sample["responses"].keys())
        order = shuffled_candidate_order(ckpts, sid, "pointwise", cfg.seed)

        for ckpt in order:
            result = judge.pointwise(
                image_path=sample["image_path"],
                question=sample["question"],
                response=sample["responses"][ckpt],
                model_label=cfg.pointwise_judge_label,
                temperature=cfg.pointwise_judge_temperature,
            )
            rows.append({
                "sample_id": sid,
                "checkpoint": ckpt,
                "score": float(result["score"]),
                "factuality": result.get("factuality"),
                "relevance": result.get("relevance"),
                "usefulness": result.get("usefulness"),
                "grounding": result.get("grounding"),
                "hallucination_penalty": result.get("hallucination_penalty"),
                "rationale": result.get("rationale", ""),
            })

    df = pd.DataFrame(rows)
    if cache_path:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache_path, index=False)
    return df


def summarize_pointwise(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("checkpoint")["score"]
        .agg(["mean", "median", "std", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )

In [ ]:
def select_after_pointwise(
    summary: pd.DataFrame,
    cfg: EvalConfig,
) -> List[str]:
    """
    Experimental default:
      retain candidates within pointwise_keep_margin of best mean,
      then cap to [min,max] candidate count.

    Replace with the historical escalation/filtering rule once recovered.
    """
    s = summary.sort_values("mean", ascending=False).copy()
    best = float(s.iloc[0]["mean"])
    kept = s[s["mean"] >= best - cfg.pointwise_keep_margin]["checkpoint"].tolist()

    if len(kept) < cfg.pointwise_min_candidates:
        kept = s.head(min(cfg.pointwise_min_candidates, len(s)))["checkpoint"].tolist()

    if len(kept) > cfg.pointwise_max_candidates:
        kept = s.head(cfg.pointwise_max_candidates)["checkpoint"].tolist()

    return kept


In [ ]:
# ---------------------------------------------------------------------
# STAGE II: LISTWISE
# ---------------------------------------------------------------------

def borda_points(ranking: Sequence[str]) -> Dict[str, int]:
    n = len(ranking)
    return {candidate: n - rank_index - 1 for rank_index, candidate in enumerate(ranking)}


def run_listwise(
    samples: Sequence[Dict[str, Any]],
    candidates: Sequence[str],
    judge: JudgeClient,
    cfg: EvalConfig,
    cache_path: Optional[str | Path] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    detail_rows = []
    score_rows = []

    for sample in samples:
        available = [c for c in candidates if c in sample["responses"]]
        order = shuffled_candidate_order(available, sample["sample_id"], "listwise", cfg.seed)

        # Per-sample temporary anonymous labels, random order:
        labels = [f"C{i+1}" for i in range(len(order))]
        label_to_ckpt = dict(zip(labels, order))
        candidate_payload = {label: sample["responses"][ckpt] for label, ckpt in label_to_ckpt.items()}

        result = judge.listwise(
            image_path=sample["image_path"],
            question=sample["question"],
            candidates=candidate_payload,
            model_label=cfg.listwise_judge_label,
            temperature=cfg.listwise_judge_temperature,
        )

        ranking_labels = result["ranking"]
        points = borda_points(ranking_labels)

        for label in ranking_labels:
            ckpt = label_to_ckpt[label]
            score_rows.append({
                "sample_id": sample["sample_id"],
                "checkpoint": ckpt,
                "borda": points[label],
                "rank": ranking_labels.index(label) + 1,
            })

        detail_rows.append({
            "sample_id": sample["sample_id"],
            "label_to_checkpoint": json.dumps(label_to_ckpt),
            "ranking_labels": json.dumps(ranking_labels),
            "rationale": json.dumps(result.get("rationale", {})),
        })

    score_df = pd.DataFrame(score_rows)
    detail_df = pd.DataFrame(detail_rows)

    if cache_path:
        base = Path(cache_path)
        base.parent.mkdir(parents=True, exist_ok=True)
        score_df.to_csv(base, index=False)
        detail_df.to_csv(base.with_name(base.stem + "_detail.csv"), index=False)

    return score_df, detail_df


def summarize_listwise(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("checkpoint")
        .agg(
            mean_borda=("borda", "mean"),
            mean_rank=("rank", "mean"),
            top1_rate=("rank", lambda x: float(np.mean(np.asarray(x) == 1))),
            count=("rank", "count"),
        )
        .sort_values(["mean_borda", "top1_rate"], ascending=[False, False])
        .reset_index()
    )

In [ ]:
# ---------------------------------------------------------------------
# STAGE III: PAIRWISE
# ---------------------------------------------------------------------

def run_pairwise(
    samples: Sequence[Dict[str, Any]],
    ckpt_a: str,
    ckpt_b: str,
    judge: JudgeClient,
    cfg: EvalConfig,
    cache_path: Optional[str | Path] = None,
) -> pd.DataFrame:
    rows = []

    for sample in samples:
        if ckpt_a not in sample["responses"] or ckpt_b not in sample["responses"]:
            continue

        # Randomize whether A or B gets ckpt_a, to reduce position bias.
        order = shuffled_candidate_order([ckpt_a, ckpt_b], sample["sample_id"], "pairwise", cfg.seed)
        shown_a, shown_b = order[0], order[1]

        result = judge.pairwise(
            image_path=sample["image_path"],
            question=sample["question"],
            response_a=sample["responses"][shown_a],
            response_b=sample["responses"][shown_b],
            model_label=cfg.pairwise_judge_label,
            temperature=cfg.pairwise_judge_temperature,
        )

        winner_label = result["winner"]
        winner_ckpt = (
            shown_a if winner_label == "A"
            else shown_b if winner_label == "B"
            else "TIE"
        )

        rows.append({
            "sample_id": sample["sample_id"],
            "ckpt_A_shown": shown_a,
            "ckpt_B_shown": shown_b,
            "winner": winner_ckpt,
            "confidence": result.get("confidence"),
            "rationale": result.get("rationale", ""),
        })

    df = pd.DataFrame(rows)
    if cache_path:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache_path, index=False)
    return df


def pairwise_win_rate(df: pd.DataFrame, ckpt_a: str, ckpt_b: str) -> Dict[str, float]:
    n = len(df)
    if n == 0:
        return {"n": 0, "a_win": np.nan, "b_win": np.nan, "tie": np.nan}
    return {
        "n": n,
        "a_win": float(np.mean(df["winner"] == ckpt_a)),
        "b_win": float(np.mean(df["winner"] == ckpt_b)),
        "tie": float(np.mean(df["winner"] == "TIE")),
    }


In [ ]:
# ---------------------------------------------------------------------
# STABILITY METRICS
# ---------------------------------------------------------------------

def bootstrap_top1_consistency(
    per_sample_df: pd.DataFrame,
    value_col: str,
    higher_is_better: bool = True,
    rounds: int = 1000,
    seed: int = 42,
) -> Dict[str, Any]:
    """
    Bootstrap samples with replacement and recompute checkpoint winner.

    Reference top-1 = winner on the full available subset.
    """
    rng = np.random.default_rng(seed)
    sample_ids = per_sample_df["sample_id"].unique()

    def rank_on(ids: Sequence[str]) -> pd.Series:
        tmp = per_sample_df[per_sample_df["sample_id"].isin(ids)]
        means = tmp.groupby("checkpoint")[value_col].mean()
        return means.sort_values(ascending=not higher_is_better)

    reference = rank_on(sample_ids)
    ref_top1 = reference.index[0]

    winners = []
    for _ in range(rounds):
        sampled = rng.choice(sample_ids, size=len(sample_ids), replace=True)
        # Preserve bootstrap multiplicities by concatenating per draw.
        pieces = []
        for j, sid in enumerate(sampled):
            x = per_sample_df[per_sample_df["sample_id"] == sid].copy()
            x["_boot_instance"] = j
            pieces.append(x)
        boot = pd.concat(pieces, ignore_index=True)
        means = boot.groupby("checkpoint")[value_col].mean()
        winner = means.idxmax() if higher_is_better else means.idxmin()
        winners.append(winner)

    consistency = float(np.mean(np.asarray(winners) == ref_top1))
    winner_counts = pd.Series(winners).value_counts(normalize=True).to_dict()

    return {
        "reference_top1": ref_top1,
        "top1_consistency": consistency,
        "winner_frequency": winner_counts,
        "rounds": rounds,
    }


In [ ]:
def bootstrap_pairwise_probability(
    df: pd.DataFrame,
    ckpt_a: str,
    ckpt_b: str,
    rounds: int = 1000,
    seed: int = 42,
) -> Dict[str, Any]:
    """
    P(A>B) under bootstrap resampling for pairwise winners.
    Ties count as 0.5 for the score difference.
    """
    rng = np.random.default_rng(seed)
    if len(df) == 0:
        return {}

    vals = np.where(
        df["winner"].to_numpy() == ckpt_a, 1.0,
        np.where(df["winner"].to_numpy() == ckpt_b, 0.0, 0.5)
    )

    stats = []
    n = len(vals)
    for _ in range(rounds):
        b = rng.choice(vals, size=n, replace=True)
        stats.append(float(np.mean(b)))

    arr = np.asarray(stats)
    return {
        "mean_bootstrap_win_score_A": float(arr.mean()),
        "P_A_gt_0.5": float(np.mean(arr > 0.5)),
        "ci95": [float(np.quantile(arr, 0.025)), float(np.quantile(arr, 0.975))],
    }


def ranking_flip_rate(
    per_sample_df: pd.DataFrame,
    value_col: str,
    rounds: int = 1000,
    seed: int = 42,
    higher_is_better: bool = True,
) -> Dict[str, Any]:
    """
    Defines a global top-1 flip rate relative to the winner on the full subset:
      flip_rate = 1 - Top1Consistency

    If your original paper used a different definition, replace this function
    with the historical implementation before reporting the metric.
    """
    result = bootstrap_top1_consistency(
        per_sample_df,
        value_col=value_col,
        higher_is_better=higher_is_better,
        rounds=rounds,
        seed=seed,
    )
    return {
        "reference_top1": result["reference_top1"],
        "flip_rate": 1.0 - result["top1_consistency"],
    }


def pairwise_ranking_agreement(
    rank_a: Sequence[str],
    rank_b: Sequence[str],
) -> float:
    """
    Fraction of checkpoint pairs whose relative ordering agrees.
    Suitable as a transparent inter-run agreement statistic.

    Use only if this matches (or replaces explicitly in a new analysis)
    the metric used in the original paper.
    """
    common = [x for x in rank_a if x in set(rank_b)]
    pos_a = {x: i for i, x in enumerate(rank_a)}
    pos_b = {x: i for i, x in enumerate(rank_b)}

    total = agree = 0
    for i in range(len(common)):
        for j in range(i + 1, len(common)):
            x, y = common[i], common[j]
            a = pos_a[x] < pos_a[y]
            b = pos_b[x] < pos_b[y]
            total += 1
            agree += int(a == b)
    return agree / total if total else float("nan")

In [ ]:
# ---------------------------------------------------------------------
# PERCENTILE-BASED SCORING + ABLATION
# ---------------------------------------------------------------------

def percentile_stability_score(values: Sequence[float], beta: float, gamma: float) -> Dict[str, float]:
    arr = np.asarray(values, dtype=float)
    p20, p50, p80 = np.percentile(arr, [20, 50, 80])
    score = p50 - beta * (p50 - p20) + gamma * (p80 - p50)
    return {
        "P20": float(p20),
        "P50": float(p50),
        "P80": float(p80),
        "score": float(score),
    }


def percentile_ablation(
    per_sample_df: pd.DataFrame,
    value_col: str,
    beta_grid: Sequence[float],
    gamma_grid: Sequence[float],
) -> pd.DataFrame:
    rows = []
    for ckpt, g in per_sample_df.groupby("checkpoint"):
        vals = g[value_col].astype(float).to_numpy()
        for beta in beta_grid:
            for gamma in gamma_grid:
                x = percentile_stability_score(vals, beta, gamma)
                rows.append({
                    "checkpoint": ckpt,
                    "beta": beta,
                    "gamma": gamma,
                    **x,
                })
    return pd.DataFrame(rows)


In [ ]:
# ---------------------------------------------------------------------
# VALIDATION / TRAINING LOSS BASELINE
# ---------------------------------------------------------------------

def validation_loss_baseline(loss_by_checkpoint: Dict[str, float]) -> Dict[str, Any]:
    """
    Lower loss is better.
    Useful external baseline for rebuttal:
    compare loss-selected checkpoint to evaluation-selected checkpoint.
    """
    if not loss_by_checkpoint:
        return {}
    ordered = sorted(loss_by_checkpoint.items(), key=lambda kv: kv[1])
    return {
        "selected_checkpoint": ordered[0][0],
        "selected_loss": float(ordered[0][1]),
        "ranking": [k for k, _ in ordered],
        "losses": {k: float(v) for k, v in ordered},
    }


# ---------------------------------------------------------------------
# OPTIONAL GENERATION-STOCHASTICITY ANALYSIS
# ---------------------------------------------------------------------

def generation_stochasticity_summary(samples: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    """
    Descriptive only unless replicate responses have already been scored.

    If response_replicates are present, reports replicate counts and exact-text
    diversity. For the rebuttal, ideally re-run pointwise/listwise evaluation
    on each generation replicate and compute top-1 consistency across replicates.
    """
    rows = []
    for s in samples:
        reps = s.get("response_replicates", {})
        for ckpt, generations in reps.items():
            if not generations:
                continue
            unique = len(set(generations))
            rows.append({
                "sample_id": s["sample_id"],
                "checkpoint": ckpt,
                "n_generations": len(generations),
                "n_unique_exact": unique,
                "exact_diversity": unique / len(generations),
            })
    return pd.DataFrame(rows)



In [ ]:
# ---------------------------------------------------------------------
# FULL AGENTIC FLOW
# ---------------------------------------------------------------------

def run_agentic_pipeline(
    samples: Sequence[Dict[str, Any]],
    judge: JudgeClient,
    cfg: EvalConfig,
    out_dir: str | Path,
) -> Dict[str, Any]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Save exact run configuration.
    save_json(asdict(cfg), out_dir / "config.json")

    # Mapping.
    mapping = build_checkpoint_mapping(samples, out_dir / "checkpoint_mapping.json", seed=cfg.seed)

    # ---------------- STAGE I ----------------
    p_df = run_pointwise(samples, judge, cfg, out_dir / "stage1_pointwise.csv")
    p_summary = summarize_pointwise(p_df)
    p_summary.to_csv(out_dir / "stage1_pointwise_summary.csv", index=False)

    kept = select_after_pointwise(p_summary, cfg)
    save_json({"kept_after_pointwise": kept}, out_dir / "stage1_selection.json")

    p_stability = bootstrap_top1_consistency(
        p_df, "score",
        higher_is_better=True,
        rounds=cfg.bootstrap_rounds,
        seed=cfg.seed,
    )
    save_json(p_stability, out_dir / "stage1_bootstrap_stability.json")

    # Percentile score + beta/gamma ablation.
    ablation = percentile_ablation(
        p_df,
        value_col="score",
        beta_grid=cfg.beta_grid,
        gamma_grid=cfg.gamma_grid,
    )
    ablation.to_csv(out_dir / "percentile_beta_gamma_ablation.csv", index=False)

    # ---------------- STAGE II ----------------
    l_df, l_detail = run_listwise(
        samples, kept, judge, cfg, out_dir / "stage2_listwise.csv"
    )
    l_summary = summarize_listwise(l_df)
    l_summary.to_csv(out_dir / "stage2_listwise_summary.csv", index=False)

    l_stability = bootstrap_top1_consistency(
        l_df, "borda",
        higher_is_better=True,
        rounds=cfg.bootstrap_rounds,
        seed=cfg.seed + 1,
    )
    save_json(l_stability, out_dir / "stage2_bootstrap_stability.json")

    finalists = l_summary.head(min(cfg.listwise_finalists, len(l_summary)))["checkpoint"].tolist()
    save_json({"finalists": finalists}, out_dir / "stage2_selection.json")


In [ ]:
# ---------------- STAGE III ----------------
    pairwise_stats = {}
    if len(finalists) >= 2:
        a, b = finalists[:2]
        pair_df = run_pairwise(
            samples, a, b, judge, cfg, out_dir / "stage3_pairwise.csv"
        )
        win = pairwise_win_rate(pair_df, a, b)
        boot = bootstrap_pairwise_probability(
            pair_df, a, b,
            rounds=cfg.bootstrap_rounds,
            seed=cfg.seed + 2,
        )
        pairwise_stats = {
            "checkpoint_A": a,
            "checkpoint_B": b,
            "raw": win,
            "bootstrap": boot,
        }
        save_json(pairwise_stats, out_dir / "stage3_pairwise_summary.json")

    # ---------------- OPTIONAL LOSS BASELINE ----------------
    # Pull a common loss dict only if present.
    loss_maps = [s.get("training_loss") for s in samples if s.get("training_loss")]
    loss_baseline = validation_loss_baseline(loss_maps[0]) if loss_maps else {}
    if loss_baseline:
        save_json(loss_baseline, out_dir / "validation_loss_baseline.json")

    # ---------------- GENERATION STOCHASTICITY ----------------
    gen_df = generation_stochasticity_summary(samples)
    if not gen_df.empty:
        gen_df.to_csv(out_dir / "generation_stochasticity_descriptive.csv", index=False)

    # ---------------- ONE REBUTTAL-FRIENDLY SUMMARY ----------------
    summary = {
        "n_samples": len(samples),
        "n_initial_checkpoints": len(mapping),
        "pointwise_top1": p_stability.get("reference_top1"),
        "pointwise_top1_consistency": p_stability.get("top1_consistency"),
        "kept_after_pointwise": kept,
        "listwise_top1": l_stability.get("reference_top1"),
        "listwise_top1_consistency": l_stability.get("top1_consistency"),
        "finalists": finalists,
        "pairwise": pairwise_stats,
        "loss_baseline": loss_baseline,
        "notes": [
            "100-sample statistics are rebuttal/public-subset analyses, not full-2400 statistics.",
            "Recover exact original R, beta, gamma, judge temperatures, and escalation thresholds before manuscript claims.",
        ],
    }
    save_json(summary, out_dir / "rebuttal_summary.json")
    return summary


In [ ]:
# ---------------------------------------------------------------------
# OPTIONAL DATA QA
# ---------------------------------------------------------------------

def validate_dataset(samples: Sequence[Dict[str, Any]]) -> None:
    if not samples:
        raise ValueError("Dataset is empty.")

    all_ckpts = [set(s.get("responses", {}).keys()) for s in samples]
    common = set.intersection(*all_ckpts) if all_ckpts else set()

    print(f"Samples: {len(samples)}")
    print(f"Checkpoints common to every sample: {len(common)}")
    print(sorted(common))

    missing_q = [s["sample_id"] for s in samples if not s.get("question")]
    missing_img = [s["sample_id"] for s in samples if not s.get("image_path")]

    if missing_q:
        print(f"WARNING: {len(missing_q)} samples missing questions.")
    if missing_img:
        print(f"WARNING: {len(missing_img)} samples missing image paths.")

    counts = pd.Series([len(s.get("responses", {})) for s in samples])
    print("Responses/sample:")
    print(counts.describe())


In [ ]:
# ---------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    p_build = sub.add_parser("build-json", help="Build JSONL from PDF pages + aligned image folder")
    p_build.add_argument("--pdf", required=True)
    p_build.add_argument("--image-folder", required=True)
    p_build.add_argument("--image-glob", default="*")
    p_build.add_argument("--out", required=True)

    p_eval = sub.add_parser("evaluate", help="Run full agentic evaluation pipeline")
    p_eval.add_argument("--data", required=True)
    p_eval.add_argument("--out-dir", required=True)
    p_eval.add_argument("--mock", action="store_true", help="Use mock judges to test pipeline")
    p_eval.add_argument("--seed", type=int, default=42)
    p_eval.add_argument("--bootstrap-rounds", type=int, default=1000)
    p_eval.add_argument("--beta", type=float, default=0.5)
    p_eval.add_argument("--gamma", type=float, default=0.25)

    args = parser.parse_args()

    if args.cmd == "build-json":
        rows = build_json_from_pdf_and_image_folder(
            pdf_path=args.pdf,
            image_folder=args.image_folder,
            output_jsonl=args.out,
            image_glob=args.image_glob,
        )
        print(f"Wrote {len(rows)} samples to {args.out}")
        return

    if args.cmd == "evaluate":
        samples = read_jsonl(args.data)
        validate_dataset(samples)

        cfg = EvalConfig(
            seed=args.seed,
            bootstrap_rounds=args.bootstrap_rounds,
            beta=args.beta,
            gamma=args.gamma,
        )

        if args.mock:
            judge = MockJudgeClient()
        else:
            raise RuntimeError(
                "Implement JudgeClient.pointwise/listwise/pairwise with your API clients, "
                "then instantiate that client here. Use --mock first to validate the pipeline."
            )

        summary = run_agentic_pipeline(samples, judge, cfg, args.out_dir)
        print(json.dumps(summary, indent=2))


In [ ]:
if __name__ == "__main__":
    main()